# LAB | sklearn Model Training + Evaluation Lab

## Telco Customer Churn

Author: Louise Plessis

Description: Predict whether customer will churn and why

In [107]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

## Step 1: Load and explore

In [108]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(df.shape)
print(df.columns.tolist())
print(df.head())
print(df.info())
print(df['Churn'].value_counts())

(7043, 21)
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             

## Step 2: Preprocessing and cleaning the data

In [109]:
# convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df['TotalCharges'].isnull().sum())   # reveals the hidden missing values

11


In [110]:
# Drop rows with missing values in TotalCharges (only 11)
df = df.dropna()

# Drop the use useless customerID column
df = df.drop('customerID', axis=1)

In [111]:
# Encode the target and text columns
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df = pd.get_dummies(df, drop_first=True)


In [112]:
# Split the data into features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

In [113]:
# Check the shape of the data

print(df.shape)
print(df.head())
print(X.shape)

(7032, 31)
   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  Churn  gender_Male  \
0              0       1           29.85         29.85      0        False   
1              0      34           56.95       1889.50      0         True   
2              0       2           53.85        108.15      1         True   
3              0      45           42.30       1840.75      0         True   
4              0       2           70.70        151.65      1        False   

   Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0         True           False             False   
1        False           False              True   
2        False           False              True   
3        False           False             False   
4        False           False              True   

   MultipleLines_No phone service  ...  StreamingTV_No internet service  \
0                            True  ...                            False   
1                           False  ...                   

## Step 3: Split the Data

In [114]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape[0], "Test:", X_test.shape[0])

Train: 5625 Test: 1407


## Step 4: Train a KNN Model

In [115]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
print("Model trained.")

Model trained.


## Step 5: Make Predictions and Evaluate

In [116]:
y_pred = knn.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7540867093105899
Precision: 0.5374331550802139
Recall: 0.5374331550802139

Confusion matrix:
[[860 173]
 [173 201]]

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.83      0.83      1033
           1       0.54      0.54      0.54       374

    accuracy                           0.75      1407
   macro avg       0.68      0.68      0.68      1407
weighted avg       0.75      0.75      0.75      1407



## Step 6: Experiment and Improve

In [117]:
for k in [1, 3, 5, 7, 9, 11, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    acc = knn.score(X_test_scaled, y_test)
    rec = recall_score(y_test, knn.predict(X_test_scaled))
    print(f"k={k}: accuracy={acc:.3f}, recall={rec:.3f}")

k=1: accuracy=0.725, recall=0.511
k=3: accuracy=0.744, recall=0.545
k=5: accuracy=0.754, recall=0.537
k=7: accuracy=0.749, recall=0.532
k=9: accuracy=0.765, recall=0.553
k=11: accuracy=0.770, recall=0.548
k=15: accuracy=0.771, recall=0.543


## Step 7: Analysis and Recommendations

### Preparation for Analysis

In [118]:
# Confirming drivers of churn by looking at the mean of tenure, MonthlyCharges, and TotalCharges for churned vs non-churned customers
df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean()


,tenure,MonthlyCharges,TotalCharges
Churn,,,
0,37.650010,61.307408,2555.344141
1,17.979133,74.441332,1531.796094


In [119]:
# Trying a different model: Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

# Train a Random Forest on the SAME split
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)   # note: no scaling needed for tree-based models

# Predict and evaluate
rf_pred = rf.predict(X_test)

print("RANDOM FOREST")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("\nClassification report:")
print(classification_report(y_test, rf_pred))

RANDOM FOREST
Accuracy: 0.7874911158493249
Recall: 0.5106951871657754

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.51      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407



In [120]:
# Feature importance from the Random Forest model
importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(10))

TotalCharges                      0.191748
tenure                            0.170417
MonthlyCharges                    0.168513
InternetService_Fiber optic       0.039125
PaymentMethod_Electronic check    0.037528
Contract_Two year                 0.030091
gender_Male                       0.029122
OnlineSecurity_Yes                0.028514
PaperlessBilling_Yes              0.025664
TechSupport_Yes                   0.024337
dtype: float64



### Analysis

#### Model Performance

I tested K values from 1 to 15. Accuracy improved slightly as K increased, peaking around **77% at k=11–15**, while recall for churners stayed roughly flat at **0.53–0.55**, with its best value at **k=9 (recall 0.553)**.

| k | Accuracy | Recall (churn) |
|----|----------|----------------|
| 1  | 0.725    | 0.511 |
| 3  | 0.744    | 0.545 |
| 5  | 0.754    | 0.537 |
| 9  | 0.765    | 0.553 |
| 11 | 0.770    | 0.548 |
| 15 | 0.771    | 0.543 |

#### Choosing the best K

If optimizing purely for accuracy, k=15 is best (0.771). However, for churn prediction the more important metric is **recall**, how many of the customers who actually churned we managed to catch, because the business goal is to identify at-risk to identify at-risk customers before they leave, so we can act to retain them.

#### The key limitation: accuracy hides a weak result

Although accuracy looks reasonable at ~77%, the model only catches around **55% of customers who actually churn**. It misses nearly half of them. This is a classic imbalanced-class problem: because most customers do not churn (~73%), a model can score high on accuracy while still performing poorly at the task that matters. For a retention use case, recall is the metric that reflects real business value, not accuracy.

#### Most important features

Grouping the data by churn confirms the main drivers:
- **Tenure**: churners average ~18 months versus ~38 months for non-churners. New customers are far more likely to leave.
- **Monthly charges**: churners pay more per month (~€74 vs ~€61), so higher-priced customers churn more.
- **Total charges**: churners have lower total charges (~€1,532 vs ~€2,555), but this mainly reflects their shorter tenure rather than being a separate cause.

The highest-risk profile is a **new customer paying a high monthly charge**.

#### Recommendations to the company

1. **Target retention efforts at high-risk segments**: short-tenure, month-to-month customers on higher monthly charges are the most likely to leave. Proactive outreach or loyalty offers should focus here.
2. **Encourage longer contracts**: since month-to-month customers churn most, incentives to move to one- or two-year contracts (discounts, added services) could directly reduce churn.
3. **Support new customers early**: because churn concentrates in low-tenure customers, a strong onboarding and early-engagement program in the first months could improve retention.

#### Limitations of the model

1. **Low recall**: the model misses ~45% of actual churners, so it is not yet reliable enough to drive retention spend on its own.
2. **KNN is not explainable**: it predicts churn but cannot tell us *why* a specific customer is at risk, which limits how actionable it is for the business.
3. **Model drift**: it is trained on past data, so it will become less accurate as customer behaviour changes and would need regular retraining.
4. **A better model likely exists**: a tree-based model such as Random Forest would probably improve performance and, importantly, provide feature-importance scores that show which factors drive churn, more useful for business decisions than KNN.

#### Feature importance (Random Forest)

The Random Forest ranked the churn drivers as follows (top features):

| Feature | Importance |
|---------|-----------|
| TotalCharges | 0.19 |
| tenure | 0.17 |
| MonthlyCharges | 0.17 |
| InternetService: Fiber optic | 0.04 |
| PaymentMethod: Electronic check | 0.04 |
| Contract: Two year | 0.03 |

The top three (total charges, tenure, monthly charges) dominate and confirm the patterns found manually: churn is driven mainly by **how long a customer has been with the company and how much they pay**. Note that TotalCharges and tenure are related (totals accumulate over time), so they partly tell the same story.

Beyond these, the model surfaced categorical drivers that the numeric averages could not:
- **Fiber-optic internet** customers churn more (higher price, higher expectations).
- **Electronic-check payers** churn more, often less-committed customers.
- **Two-year contracts** are associated with lower churn, confirming that longer contracts retain customers.

(Gender appeared with small importance but is unlikely to be a genuine driver, probably statistical noise, so I would not act on it.)

## Bonus

In [121]:
## Bonus Challenge 3: Comparing Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Logistic Regression (linear model, likes scaled data)
logreg = LogisticRegression(max_iter=5000)
logreg.fit(X_train_scaled, y_train)
logreg_pred = logreg.predict(X_test_scaled)

# Support Vector Classifier (also distance-based, needs scaling)
svc = SVC()
svc.fit(X_train_scaled, y_train)
svc_pred = svc.predict(X_test_scaled)

# Compare all on recall and accuracy
print("Logistic Regression -> accuracy:", round(accuracy_score(y_test, logreg_pred), 3),
      "recall:", round(recall_score(y_test, logreg_pred), 3))
print("SVC                 -> accuracy:", round(accuracy_score(y_test, svc_pred), 3),
      "recall:", round(recall_score(y_test, svc_pred), 3))

Logistic Regression -> accuracy: 0.804 recall: 0.575
SVC                 -> accuracy: 0.787 recall: 0.492


#### Bonus Challenge 3: Algorithm Comparison

I compared four classifiers on the same train/test split:

| Model | Accuracy | Recall (churn) |
|-------|----------|----------------|
| **Logistic Regression** | **0.804** | **0.575** |
| Random Forest | 0.787 | 0.511 |
| KNN (k=9) | 0.765 | 0.553 |
| SVC | 0.787 | 0.492 |

**Logistic Regression performed best on both accuracy and recall.** This is a useful and slightly counterintuitive result: the simplest, most explainable model outperformed the more complex ones (Random Forest, SVC) on this tabular dataset. This is common with structured business data, where a well-chosen linear model is often hard to beat.

Logistic Regression is also the most **interpretable** of the four (each feature has a clear weight showing its influence on churn), which makes it not just the best performer here but also the most useful for a business that needs to understand *why* customers are flagged. For this churn use case it is my recommended model.

It is worth noting that all four models still cap out at ~0.49–0.58 recall. The choice of algorithm makes only a modest difference; the real ceiling is the class imbalance

In [122]:
## Bonus Challenge 5: Business Impact Analysis
from sklearn.metrics import confusion_matrix

# Get the four outcomes for Logistic Regression (our best model)
cm = confusion_matrix(y_test, logreg_pred)
TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

# Assign business costs (these are assumptions - state them clearly)
COST_FALSE_NEGATIVE = 500   # € lost when we miss a churner (lost customer value)
COST_FALSE_POSITIVE = 50    # € wasted on a retention offer to someone who'd have stayed

# Total cost of the model's mistakes
cost_fn = FN * COST_FALSE_NEGATIVE
cost_fp = FP * COST_FALSE_POSITIVE
total_cost = cost_fn + cost_fp

print(f"Missed churners (FN): {FN}  ->  cost: €{cost_fn:,}")
print(f"Wasted offers   (FP): {FP}  ->  cost: €{cost_fp:,}")
print(f"Total cost of model errors: €{total_cost:,}")

Missed churners (FN): 159  ->  cost: €79,500
Wasted offers   (FP): 117  ->  cost: €5,850
Total cost of model errors: €85,350


#### Bonus Challenge 5: Business Impact Analysis

To translate model performance into business terms, I assigned a euro cost to each type of error:
- **False negative** (missed churner → lost customer): **€500**
- **False positive** (unnecessary retention offer): **€50**

These are assumptions, but they reflect a key reality: losing a customer costs far more than a wasted retention offer (here, ~10x more).

**Results for the Logistic Regression model:**

| Error type | Count | Cost |
|------------|-------|------|
| Missed churners (FN) | 159 | €79,500 |
| Wasted offers (FP) | 117 | €5,850 |
| **Total cost of errors** | | **€85,350** |

**Key insight:** the missed churners account for **93% of the total error cost** (€79,500 of €85,350), even though there are only slightly more of them than wasted offers. This confirms in euros what the recall metric suggested: for churn, **false negatives are the expensive mistake**, and the model's biggest weakness (catching only ~58% of churners) is also its costliest one.

**Business implication:** because a missed churner costs 10x a wasted offer, the company should prefer a model that catches more churners even at the price of more false positives. It is cheaper to send a few unnecessary retention offers than to lose customers. This means the model should be tuned to **maximise recall** (e.g. by lowering the decision threshold or addressing class imbalance), accepting lower precision as a worthwhile trade.